# Chapitre 0.2 : les maths utiles

Une IA ne fait que des calculs simples, mais des milliards de fois. Dans ce carnet, tu fais tourner les quelques maths dont elle a besoin, dans un vrai petit programme : des vecteurs, des matrices, une pente qu’on descend, des tirages au sort, puis des probabilités et une erreur.

Ce carnet accompagne [la page du chapitre 0.2](https://chipat-neko.github.io/forge-ia/partie-0/0-2/) : tu peux la lire avant, ou en même temps.

**Ce que tu vas faire**

1. calculer le produit scalaire de deux vecteurs, et voir s’ils vont dans le même sens ;
2. multiplier une matrice par un vecteur ;
3. mesurer la pente d’une erreur, exacte puis approchée ;
4. descendre cette pente à plusieurs vitesses, et voir quand ça marche et quand ça explose ;
5. tirer au sort selon des probabilités, avec un hasard qu’on peut rejouer ;
6. changer des scores en probabilités avec softmax, puis mesurer l’erreur d’une prédiction ;
7. relever un défi, puis vérifier que ce carnet trouve exactement les mêmes nombres que la page du site.

**Comment l’utiliser**

Il n’y a rien à installer. Lance les cellules de code de haut en bas : clique sur le bouton rond avec un triangle, à gauche de la cellule, ou tape Maj + Entrée. Tu peux aussi choisir « Exécution », puis « Tout exécuter ». Tu peux modifier le code et le relancer autant de fois que tu veux : tu ne peux rien casser.

## Étape 0 : récupérer le code

Le code du projet est dans un dépôt GitHub public. La cellule suivante le télécharge avec `git clone`, puis indique à Python où trouver le projet. Le projet n’utilise que ce qui est livré avec Python : il n’y a rien d’autre à installer.

Elle définit aussi `nombre`, une petite fonction qui écrit les nombres à la française, avec une virgule : `nombre(0.1)` donne 0,1.

In [ ]:
import os
import sys
from pathlib import Path

# Sur Colab, on télécharge le dépôt public. Quand ce carnet est lancé par les tests
# du projet, la variable FORGE_IA_LOCAL donne le chemin d’un dépôt déjà présent :
# on l’utilise, sans rien télécharger.
depot_local = os.environ.get("FORGE_IA_LOCAL")
if depot_local:
    depot = Path(depot_local)
else:
    depot = Path("forge-ia")
    if not depot.exists():
        !git clone --depth 1 https://github.com/chipat-neko/forge-ia

projet = depot / "projets" / "0.2-maths"
if not projet.exists():
    raise FileNotFoundError("Le dépôt n’a pas pu être téléchargé. Vérifie ta connexion, puis relance cette cellule.")

# Python cherche les modules dans les dossiers de sys.path : on y ajoute le projet.
sys.path.insert(0, str(projet.resolve()))


def nombre(x, decimales=None):
    # Un nombre à la française. « :, » sépare les milliers par des virgules, à l’anglaise :
    # on les remplace par l’espace fine (\u202f), puis le point par une virgule.
    # Sans nombre de décimales, juste ce qu’il faut (6 au plus) : 0,1 et pas 0,100000.
    x = x + 0.0  # -0.0 + 0.0 donne 0.0 : jamais « -0 »
    texte = f"{x:,.{6 if decimales is None else decimales}f}"
    if decimales is None:
        texte = texte.rstrip("0").rstrip(".")
    return texte.replace(",", "\u202f").replace(".", ",")


print("Le code du projet est prêt.")

## Étape 1 : des vecteurs

Un **vecteur** est une liste de nombres. Dans le plan, `[3, 1]` est une flèche qui va de 3 vers la droite et de 1 vers le haut. Un mot, pour ton IA, est un vecteur de plusieurs centaines de nombres : on ne peut plus le dessiner, mais tout ce qui suit reste vrai.

Le **produit scalaire** de deux vecteurs multiplie leurs nombres case par case, puis additionne : a · b = 3 × 1 + 1 × 2 = 5. Il est grand quand les deux flèches vont dans le même sens, nul quand elles sont perpendiculaires, négatif quand elles s’opposent.

Le **cosinus** retire la longueur des flèches : c’est le produit scalaire divisé par les deux longueurs. Il va de −1 (sens opposés) à 1 (même direction). Un vecteur nul n’a pas de direction : `cosinus` renvoie alors `None`, « rien ».

Change les nombres de `a` et `b`, puis relance la cellule.

In [ ]:
from maths_utiles.vecteurs import cosinus, produit_scalaire, somme

a = [3, 1]
b = [1, 2]

print(f"a · b = {a[0]} × {b[0]} + {a[1]} × {b[1]} = {nombre(produit_scalaire(a, b))}")
print(f"a + b = {somme(a, b)}")
print(f"Cosinus de l’angle entre a et b\u202f: {nombre(cosinus(a, b), 3)}")

# Deux vecteurs perpendiculaires : leur produit scalaire vaut 0.
c = [2, 1]
d = [-1, 2]
print(f"c · d = {nombre(produit_scalaire(c, d))}, cosinus = {nombre(cosinus(c, d), 3)}")

# Le vecteur nul n’a pas de direction.
print(f"Cosinus avec le vecteur nul\u202f: {cosinus(a, [0, 0])}")

## Étape 2 : une matrice fois un vecteur

Une **matrice** est un tableau de nombres, rangé en lignes : en Python, une liste de lignes. La multiplier par un vecteur, c’est faire un produit scalaire par ligne : chaque ligne de la matrice, fois le vecteur, donne une case du résultat.

Une couche de ton IA est une matrice : elle change le vecteur d’un mot en un autre vecteur, avec un produit scalaire par ligne. Il faut donc que chaque ligne ait autant de nombres que le vecteur. Sinon, la fonction s’arrête avec un message qui dit pourquoi. Le bloc `try` / `except` attrape cette erreur pour l’afficher, au lieu d’arrêter la cellule.

In [ ]:
from maths_utiles.vecteurs import matrice_fois_vecteur

m = [
    [1, 2, 3],
    [0, 1, 0],
]
v = [1, 1, 1]
# Une case par ligne : [1, 2, 3] · v, puis [0, 1, 0] · v.
print(matrice_fois_vecteur(m, v))

# Une ligne de 4 nombres ne se multiplie pas par un vecteur de 3 nombres.
try:
    matrice_fois_vecteur([[1, 2, 3, 4]], [1, 2, 3])
except ValueError as probleme:
    print(f"Erreur\u202f: {probleme}")

## Étape 3 : la pente

Une IA apprend en réglant ses poids pour que son erreur baisse. Prenons un seul poids, `x`, et une erreur toute simple : (x − 3)² + 1. Elle est la plus petite, 1, quand x vaut 3.

La **pente** de l’erreur en `x` dit de combien elle monte quand `x` avance un tout petit peu. Ici, elle vaut 2 × (x − 3) : négative à gauche de 3, où l’erreur descend vers la droite, positive à droite, nulle au fond.

Sans connaître la formule, on peut aussi l’**approcher** : regarder l’erreur un tout petit peu à droite et à gauche de `x`, à un millionième près, et diviser la montée par la distance. Les deux pentes sont presque égales : c’est ainsi qu’on vérifie une formule de pente.

In [ ]:
from maths_utiles.derivees import erreur, pente, pente_approchee

for x in [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0]:
    print(f"x = {nombre(x)}\u202f: erreur = {nombre(erreur(x))}, pente = {nombre(pente(x))}")

print(f"Pente exacte en x = 0\u202f: {pente(0.0)}")
print(f"Pente approchée en x = 0\u202f: {pente_approchee(erreur, 0.0)}")

## Étape 4 : descendre la pente

Pour faire baisser l’erreur, on fait un pas contre la pente : x ← x − taux × pente(x). Si la pente est négative, `x` avance vers la droite ; si elle est positive, il recule. Puis on recommence. C’est la **descente de gradient**, et c’est ainsi qu’apprend ton IA, sur des centaines de millions de poids à la fois.

Le **taux** règle la taille des pas. La cellule suivante part de x = 0, au taux 0,1, et montre les dix premiers pas : x se rapproche de 3, et l’erreur de 1.

In [ ]:
from maths_utiles.derivees import descendre

for n, x in enumerate(descendre(0.0, 0.1, 10)):
    print(f"Pas {n}\u202f: x = {nombre(x, 3)}, erreur = {nombre(erreur(x), 3)}, pente = {nombre(pente(x), 3)}")

Avec un taux plus grand, la descente va plus vite… jusqu’à un certain point. La cellule suivante essaie chaque taux de la figure du chapitre, et cherche au bout de combien de pas l’erreur passe sous 1,01, tout près du fond.

Au taux 1, `x` saute d’un bord à l’autre sans jamais descendre : 0, 6, 0, 6… Au-delà, chaque pas l’éloigne un peu plus du fond : la descente explose.

In [ ]:
from maths_utiles.derivees import TAUX_DESCENTE, pas_pour_atteindre

for taux in TAUX_DESCENTE:
    n = pas_pour_atteindre(0.0, taux, 1.01)
    if n is None:
        fin = descendre(0.0, taux, 30)[-1]
        print(f"Au taux {nombre(taux)}, l’erreur ne passe jamais sous 1,01. Après 30 pas, x = {nombre(fin, 3)}.")
    else:
        print(f"Au taux {nombre(taux)}, l’erreur passe sous 1,01 au pas {n}.")

**La règle de la chaîne.** Quand une fonction en enchaîne deux, sa pente est le produit de leurs pentes. Par exemple, h(x) = (2x + 1)² applique d’abord f(x) = 2x + 1, dont la pente vaut 2, puis g(u) = u², dont la pente vaut 2u. La pente de h est donc 2 × (2x + 1), fois 2 : en x = 1, elle vaut 12. Une IA enchaîne des centaines d’opérations : c’est cette règle qui lui donne la pente de son erreur selon chacun de ses poids. La pente approchée le confirme.

In [ ]:
from maths_utiles.derivees import exemple_chaine

chaine = exemple_chaine()
print(f"h(1) = {nombre(chaine['h'])}")
print(f"Pente de h en 1, par la règle de la chaîne\u202f: {nombre(chaine['exacte'])}")
print(f"Pente de h en 1, approchée\u202f: {chaine['approchee']}")

## Étape 5 : tirer au sort

Dans un petit texte, le mot « le » est suivi 6 fois de « fer », 3 fois de « métal » et 1 fois de « feu ». Chaque compte divisé par le total donne une **probabilité** : 0,6, 0,3 et 0,1.

Pour tirer au sort selon ces probabilités, on les pose bout à bout de 0 à 1, puis on prend un nombre `u` au hasard entre 0 et 1 : il tombe dans l’une d’elles. Le hasard d’un ordinateur est un calcul : `Hasard` part d’une **graine**, un entier, et en tire une suite de nombres qui a l’air prise au hasard. La même graine redonne toujours la même suite, ici comme dans la page du site.

In [ ]:
from maths_utiles.tirage import EXEMPLE_COMPTES, GRAINE_TIRAGE, Hasard, probabilites, tirer

mots = list(EXEMPLE_COMPTES)
probas = probabilites(EXEMPLE_COMPTES)
for mot, p in zip(mots, probas):
    print(f"«\u202f{mot}\u202f», {EXEMPLE_COMPTES[mot]} fois sur 10\u202f: probabilité {nombre(p)}")

hasard = Hasard(GRAINE_TIRAGE)
for _ in range(5):
    u = hasard.uniforme()
    print(f"u = {nombre(u, 3)} tombe sur «\u202f{mots[tirer(probas, u)]}\u202f»")

Sur peu de tirages, les fréquences s’écartent des probabilités ; sur beaucoup, elles s’en rapprochent. Les probabilités ne se voient qu’en moyenne. Change la graine, puis relance : les petits nombres de tirages changent beaucoup, les grands presque pas.

In [ ]:
from maths_utiles.tirage import tirages

graine = GRAINE_TIRAGE
for n in [10, 100, 1_000, 10_000, 100_000]:
    comptes = tirages(probas, n, graine)
    frequences = ", ".join(f"{mot} {nombre(100 * c / n, 1)}\u202f%" for mot, c in zip(mots, comptes))
    print(f"Sur {nombre(n)} tirages\u202f: {frequences}")

## Étape 6 : des scores aux probabilités

Une IA ne sort pas directement des probabilités : elle donne un **score** à chaque réponse possible, un nombre quelconque. **Softmax** change ces scores en probabilités, positives et de somme 1 : elle prend l’exponentielle de chaque score, puis divise par la somme.

La **température** divise les scores avant l’exponentielle. Basse, elle avantage le plus grand score : l’IA choisit presque toujours la même réponse. Haute, elle rapproche les probabilités : l’IA prend plus de risques.

In [ ]:
from maths_utiles.probabilites import softmax

scores = [2.0, 1.0, 0.1]
for temperature in [0.5, 1.0, 2.0]:
    probas_softmax = ", ".join(nombre(p, 3) for p in softmax(scores, temperature))
    print(f"Température {nombre(temperature)}\u202f: {probas_softmax}")

## Étape 7 : l’erreur d’une prédiction

Pour apprendre, une IA mesure son erreur sur chaque prédiction : −ln p, où p est la probabilité qu’elle donnait à la bonne réponse. Sûre et juste (p = 1), son erreur est nulle. Plus elle donnait peu de chances à la bonne réponse, plus l’erreur est grande.

Un modèle qui devine au hasard parmi 40 caractères, comme le mini-modèle du site, donne 1 chance sur 40 à chacun : son erreur de départ vaut ln 40.

In [ ]:
import math

from maths_utiles.probabilites import erreur_de_prediction

for p in [1.0, 0.5, 0.1, 1 / 40]:
    print(f"p = {nombre(p, 3)}\u202f: erreur = {nombre(erreur_de_prediction(p), 3)}")
print(f"ln 40 = {nombre(math.log(40), 3)}")

## Exercice : la longueur d’un vecteur

Écris la fonction `norme(a)`. Elle reçoit un vecteur, et renvoie sa **longueur** : la racine carrée du produit scalaire du vecteur avec lui-même. `math.sqrt` donne la racine carrée.

La cellule essaie ta fonction sur le vecteur `[3, 4]`. C’est le grand côté d’un triangle rectangle de côtés 3 et 4 : sa longueur vaut 5. Si ta fonction trouve 5, le défi est réussi.

Indice : `produit_scalaire(a, a)` vaut 3 × 3 + 4 × 4.

In [ ]:
import math

from maths_utiles.vecteurs import produit_scalaire


def norme(a: list[float]) -> float:
    # À toi : la racine carrée du produit scalaire de a avec lui-même.
    return 0.0


# Ne change rien en dessous : cette partie essaie ta fonction sur le vecteur [3, 4].
if norme([3, 4]) == 5:
    print("Défi réussi\u202f: la longueur de [3, 4] vaut 5.")
else:
    print(f"Pas encore. Ta fonction donne {norme([3, 4])} pour [3, 4], au lieu de 5.")

<details>
<summary>Voir une correction (essaie d’abord !)</summary>

Le produit scalaire de `a` avec lui-même additionne le carré de chaque nombre : 3 × 3 + 4 × 4 = 25. Sa racine carrée vaut 5.

Remplace la fonction de la cellule de l’exercice par :

```python
def norme(a: list[float]) -> float:
    return math.sqrt(produit_scalaire(a, a))
```

</details>

## Dernière étape : les mêmes nombres que la page ?

La page du site n’utilise pas Python : ses figures refont les vecteurs, la descente et les tirages en TypeScript, directement dans ton navigateur. Pour être sûr que les deux donnent exactement les mêmes nombres, le dépôt garde un fichier de référence, `tests/attendus.json`, calculé une fois pour toutes par le Python. Les tests de la page relisent ce fichier. La cellule suivante recalcule tout avec les fonctions du projet, et compare au bit près chaque nombre qui ne demande que des additions, des soustractions, des multiplications, des divisions et des racines carrées, ainsi que les messages d’erreur, que la page affiche aussi.

Softmax et l’erreur −ln p passent par l’exponentielle et le logarithme, dont le dernier chiffre peut changer d’un ordinateur à l’autre : la page du site ne les recalcule pas, elle lit les valeurs du fichier.

In [ ]:
import json

from maths_utiles.attendus import SECTIONS_EXACTES, calculer_attendus, vers_json

attendus = json.loads((projet / "tests" / "attendus.json").read_text(encoding="utf-8"))
calcule = calculer_attendus()

print(
    f"{len(attendus['vecteurs'])} paires de vecteurs, {len(attendus['matrices'])} matrices, "
    f"{len(attendus['descente']['cas'])} descentes et {len(attendus['tirage']['tirages'])} séries de tirages comparées."
)
# Chaque section est écrite en JSON, comme dans le fichier, puis comparée caractère par
# caractère : « == » prendrait -0.0 pour 0.0, et 6 pour 6.0.
differentes = [nom for nom in SECTIONS_EXACTES if vers_json(calcule[nom]) != vers_json(attendus[nom])]
for nom in differentes:
    print(f"La section «\u202f{nom}\u202f» n’est pas celle du fichier de référence.")

print(f"Même résultat que la page du site\u202f: {'oui' if not differentes else 'non'}")  # \u202f est l’espace fine insécable

## Pour aller plus loin

- Le code complet du projet, avec ses tests : [projets/0.2-maths sur GitHub](https://github.com/chipat-neko/forge-ia/tree/main/projets/0.2-maths).
- La page du chapitre, pour manipuler les trois figures, faire les paris et lire le code expliqué : [chapitre 0.2](https://chipat-neko.github.io/forge-ia/partie-0/0-2/).
- Le chapitre 0.3 prépare ton ordinateur : Python, uv et PyTorch, et la carte graphique si tu en as une.

Le code de ce carnet est sous licence MIT, ses textes sous licence CC BY 4.0.